<a href="https://colab.research.google.com/github/jimmyGit538/coin-market-cap-project/blob/production/Jessica/CMC_api_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#remove this for couldrun

from google.colab import auth
auth.authenticate_user()
print("Authenticated")


Authenticated


In [ ]:

# Install and import libraries


!pip install --quiet requests pandas google-cloud-bigquery db-dtypes

import requests
import pandas as pd
from datetime import datetime, timedelta
from getpass import getpass

from google.cloud import bigquery


# Configuration section
#    Edit only if names change


# GCP project
PROJECT_ID = "coinmarketcapproject"

# Dataset where new tables will live
DATASET_ID = "crypto_raw"

# New destination tables
TABLE_MAP_NEW = "map_gcp_raw"
TABLE_CATEGORIES_NEW = "categories_gcp_raw"
TABLE_CATEGORY_TOP20 = "category_top20_raw"
TABLE_CATEGORY_TOP20_COINS = "category_top20_coins_raw"   # <-- ADD THIS
TABLE_LISTINGS_HISTORICAL = "listings_historical_daily_raw"


# CoinMarketCap API configuration
BASE_URL = "https://pro-api.coinmarketcap.com/v1"

# In Colab this prompts you securely.
# In Cloud Run you replace this with an environment variable.
CMC_API_KEY = getpass("Enter your CoinMarketCap API key: ").strip()

# Common request headers for CMC
HEADERS = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": CMC_API_KEY
}

# BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)


Enter your CoinMarketCap API key: ··········


In [ ]:
# Helper functions


def call_cmc(endpoint, params=None):
    """
    Calls a CoinMarketCap endpoint and returns the "data" portion of the JSON response.

    endpoint: string such as "cryptocurrency/map"
    params: dictionary of query parameters for the API
    """
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, headers=HEADERS, params=params or {})
    response.raise_for_status()  # raises a clear error if status code is not 200
    json_response = response.json()
    return json_response.get("data", [])


def write_df_to_bigquery(df, table_name, write_disposition="WRITE_APPEND"):
    """
    Writes a pandas DataFrame to a BigQuery table, including nested fields
    like lists of dictionaries (REPEATED RECORD).
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition,
        autodetect=True,                      # allow schema inference
        source_format=bigquery.SourceFormat.PARQUET  # <-- KEY FIX
    )

    # BigQuery handles nested lists better when using PARQUET format
    load_job = bq_client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config
    )

    load_job.result()
    print(f"Loaded {len(df)} rows into {table_id}")



In [ ]:
import time
from requests.exceptions import HTTPError


In [ ]:
# MAP endpoint


def pull_all_map(listing_status="active", sort="cmc_rank", page_size=5000):
    """
    Pulls all rows from the CoinMarketCap cryptocurrency map endpoint
    using pagination.

    listing_status: for example "active"
    sort: for example "cmc_rank"
    page_size: number of rows per request, CMC usually allows up to 5000
    """
    all_rows = []
    start = 1

    while True:
        print(f"Pulling map data starting at {start}")

        params = {
            "listing_status": listing_status,
            "sort": sort,
            "start": start,
            "limit": page_size
        }

        data_chunk = call_cmc(
            endpoint="cryptocurrency/map",
            params=params
        )

        if not data_chunk:
            print("No more data returned. Stopping.")
            break

        all_rows.extend(data_chunk)

        # If we got less than the page size, that means we reached the end
        if len(data_chunk) < page_size:
            print("Last page received. Stopping.")
            break

        # Move the start pointer forward for the next request
        start += page_size

    return all_rows


# Call the map endpoint and get all rows
map_data = pull_all_map(
    listing_status="active",
    sort="cmc_rank",
    page_size=5000
)

# Convert to DataFrame
df_map = pd.DataFrame(map_data)

# Add ingestion timestamp
df_map["ingestion_timestamp_utc"] = pd.Timestamp.utcnow()

# Write to BigQuery
write_df_to_bigquery(
    df_map,
    TABLE_MAP_NEW,
    write_disposition="WRITE_APPEND"
)



Pulling map data starting at 1
Pulling map data starting at 5001
Pulling map data starting at 10001
Last page received. Stopping.
Loaded 10059 rows into coinmarketcapproject.crypto_raw.map_gcp_raw


In [ ]:
# CATEGORIES endpoint

def pull_all_categories():
    """
    Calls the CoinMarketCap categories endpoint once and
    returns the full data payload.

    The endpoint already returns all categories your plan
    is allowed to see, so we don't need pagination.
    """
    return call_cmc(endpoint="cryptocurrency/categories")


# pull all categories
categories_data = pull_all_categories()

# convert to DataFrame
df_categories = pd.DataFrame(categories_data)

# timestamp
df_categories["ingestion_timestamp_utc"] = pd.Timestamp.utcnow()

# write to BigQuery
write_df_to_bigquery(df_categories, TABLE_CATEGORIES_NEW, "WRITE_APPEND")


Loaded 338 rows into coinmarketcapproject.crypto_raw.categories_gcp_raw


In [ ]:
# CATEGORY endpoint (detail for top 20 categories)


from pandas import json_normalize
import pandas as pd

top20_category_ids = (
    df_categories
    .sort_values("market_cap", ascending=False)
    .head(20)["id"]
    .tolist()
)

category_frames = []

for cat_id in top20_category_ids:
    print(f"Pulling detail for category {cat_id}")

    cat_data = call_cmc(
        endpoint="cryptocurrency/category",
        params={"id": cat_id}
    )

    df_cat = json_normalize(cat_data)

    # keep track of which category this came from
    df_cat["source_category_id"] = cat_id
    df_cat["ingestion_timestamp_utc"] = pd.Timestamp.utcnow()

    category_frames.append(df_cat)

# combine all 20 into one DataFrame
df_category_top20 = pd.concat(category_frames, ignore_index=True)

# FIX: convert nested "coins" list into a STRING column
if "coins" in df_category_top20.columns:
    df_category_top20["coins"] = df_category_top20["coins"].astype(str)

# write to BigQuery
write_df_to_bigquery(
    df_category_top20,
    TABLE_CATEGORY_TOP20,
    "WRITE_APPEND"
)



Pulling detail for category 6825fdf884b53e60df77be0a
Pulling detail for category 6433de7df79a2653906cd680
Pulling detail for category 6437284985f6a3507d5fd57d
Pulling detail for category 605e2cc16507f27280c38980
Pulling detail for category 67c514446feebc2b5bcc23f1
Pulling detail for category 605e2e73d41eae1066535f80
Pulling detail for category 605e2e38d41eae1066535f7f
Pulling detail for category 605e2ce9d41eae1066535f7c
Pulling detail for category 604f2772ebccdd50cd175fd9
Pulling detail for category 605e2a01d41eae1066535f71
Pulling detail for category 605e2b07d41eae1066535f75
Pulling detail for category 604f2775ebccdd50cd175fdb
Pulling detail for category 605e2a98d41eae1066535f74
Pulling detail for category 6051abf38a9b3f285eec4d3f
Pulling detail for category 605e29ad6507f27280c3897c
Pulling detail for category 604f2774ebccdd50cd175fda
Pulling detail for category 605e2acf6507f27280c3897d
Pulling detail for category 605e2b4dd41eae1066535f76
Pulling detail for category 605e2967d41eae1066

In [ ]:
import pandas as pd
import time # Import time module for delays

coin_rows = []

for cat_id in top20_category_ids:
    print(f"Pulling coins for category {cat_id}")

    cat_data = call_cmc(
        endpoint="cryptocurrency/category",
        params={"id": cat_id}
    )

    coins = cat_data.get("coins", [])

    for coin in coins:   # <-- NOW inside the category loop
        usd = (coin.get("quote") or {}).get("USD", {})

        coin_rows.append({
            "category_id": cat_id,
            "coin_id": coin.get("id"),
            "coin_name": coin.get("name"),
            "coin_symbol": coin.get("symbol"),
            "coin_slug": coin.get("slug"),
            "coin_cmc_rank": coin.get("cmc_rank"),
            "coin_market_cap": usd.get("market_cap"),
            "coin_price": usd.get("price"),
            "coin_volume_24h": usd.get("volume_24h"),
            "ingestion_timestamp_utc": pd.Timestamp.utcnow(),
        })

    # Add a delay to avoid rate limiting
    time.sleep(1.5)

# turn list of dicts into DataFrame
df_category_coins = pd.DataFrame(coin_rows)


# write to BigQuery
write_df_to_bigquery(
    df_category_coins,
    TABLE_CATEGORY_TOP20_COINS,   # "category_top20_coins_raw"
    "WRITE_APPEND"
)


Pulling coins for category 6825fdf884b53e60df77be0a
Pulling coins for category 6433de7df79a2653906cd680
Pulling coins for category 6437284985f6a3507d5fd57d
Pulling coins for category 605e2cc16507f27280c38980
Pulling coins for category 67c514446feebc2b5bcc23f1
Pulling coins for category 605e2e73d41eae1066535f80
Pulling coins for category 605e2e38d41eae1066535f7f
Pulling coins for category 605e2ce9d41eae1066535f7c
Pulling coins for category 604f2772ebccdd50cd175fd9
Pulling coins for category 605e2a01d41eae1066535f71
Pulling coins for category 605e2b07d41eae1066535f75
Pulling coins for category 604f2775ebccdd50cd175fdb
Pulling coins for category 605e2a98d41eae1066535f74
Pulling coins for category 6051abf38a9b3f285eec4d3f
Pulling coins for category 605e29ad6507f27280c3897c
Pulling coins for category 604f2774ebccdd50cd175fda
Pulling coins for category 605e2acf6507f27280c3897d
Pulling coins for category 605e2b4dd41eae1066535f76
Pulling coins for category 605e2967d41eae1066535f70
Pulling coin

In [ ]:
for cat_id in top20_category_ids:
    cat_data = call_cmc(
        endpoint="cryptocurrency/category",
        params={"id": cat_id}
    )
    coins = cat_data.get("coins", [])
    print(cat_id, "coins:", len(coins))


6825fdf884b53e60df77be0a coins: 100
6433de7df79a2653906cd680 coins: 100
6437284985f6a3507d5fd57d coins: 27
605e2cc16507f27280c38980 coins: 59
67c514446feebc2b5bcc23f1 coins: 5
605e2e73d41eae1066535f80 coins: 33
605e2e38d41eae1066535f7f coins: 61
605e2ce9d41eae1066535f7c coins: 27
604f2772ebccdd50cd175fd9 coins: 61
605e2a01d41eae1066535f71 coins: 37
605e2b07d41eae1066535f75 coins: 18
604f2775ebccdd50cd175fdb coins: 46
605e2a98d41eae1066535f74 coins: 26
6051abf38a9b3f285eec4d3f coins: 84
605e29ad6507f27280c3897c coins: 16
604f2774ebccdd50cd175fda coins: 34
605e2acf6507f27280c3897d coins: 28
605e2b4dd41eae1066535f76 coins: 21
605e2967d41eae1066535f70 coins: 21
605e2e0bd41eae1066535f7e coins: 19
